# Morphology-aware Old Tupi tokenizer/canonicalizer proof of concept

This notebook sketches a tokenizer that learns from the pydicate + oldtupicorpus ecosystem instead of treating Old Tupi as anonymous text.

The target shape is:

```text
raw Old Tupi surface text
-> normalization and orthography handling
-> morpheme / allomorph segmentation
-> grammar-aware canonical stream
-> reversible-ish inspection
```

This is not ordinary BPE. BPE would learn frequent string fragments, but it would not know that `r` can be a pluriform prefix, that `oré` may carry person/role features, or that annotated/generated pydicate data can teach a canonical morphology stream. Here the pydicate-rendered corpus is the teacher.

## 1. Setup

Run this notebook from the `oldtupicorpus` repo root. The setup cell searches upward for the repo root, then adds:

- the repo root itself
- `../nhe-enga/tupi`
- `../nhe-enga/pydicate`

The imports are intentionally checked up front because rebuilding corpus artifacts depends on the local sibling repo.

In [2]:
from __future__ import annotations

import json
import math
import os
import random
import re
import subprocess
import sys
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "tokenizer").is_dir() and (candidate / "historic").is_dir():
            return candidate
    raise RuntimeError("Could not find oldtupicorpus repo root from current directory")


ROOT = find_repo_root()
NHE_ENGA = (ROOT.parent / "nhe-enga").resolve()
PATHS_TO_ADD = [ROOT, NHE_ENGA / "tupi", NHE_ENGA / "pydicate"]

for path in reversed(PATHS_TO_ADD):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f"Repo root: {ROOT}")
for path in PATHS_TO_ADD:
    print(f"sys.path entry {'OK' if path.exists() else 'MISSING'}: {path}")

IMPORT_STATUS = {}

try:
    import tupi
    from tupi import TupiAntigo
    IMPORT_STATUS["tupi"] = True
    print(f"OK: import tupi -> {getattr(tupi, '__file__', '(namespace package)')}")
except Exception as exc:
    IMPORT_STATUS["tupi"] = False
    print(f"FAIL: import tupi: {type(exc).__name__}: {exc}")

try:
    from pydicate.lang.tupilang import *  # noqa: F401,F403
    IMPORT_STATUS["pydicate.lang.tupilang"] = True
    print("OK: from pydicate.lang.tupilang import *")
except Exception as exc:
    IMPORT_STATUS["pydicate.lang.tupilang"] = False
    print(f"FAIL: from pydicate.lang.tupilang import *: {type(exc).__name__}: {exc}")

try:
    from pydicate.lang.tupilang.pos import *  # noqa: F401,F403
    IMPORT_STATUS["pydicate.lang.tupilang.pos"] = True
    print("OK: from pydicate.lang.tupilang.pos import *")
except Exception as exc:
    IMPORT_STATUS["pydicate.lang.tupilang.pos"] = False
    print(f"FAIL: from pydicate.lang.tupilang.pos import *: {type(exc).__name__}: {exc}")

if not all(IMPORT_STATUS.values()):
    print("\nSome imports failed. Existing tokenizer/output artifacts can still be inspected, but rebuilding them may fail until the sibling repo is importable.")

Repo root: /Users/kian/code/oldtupicorpus
sys.path entry OK: /Users/kian/code/oldtupicorpus
sys.path entry OK: /Users/kian/code/nhe-enga/tupi
sys.path entry OK: /Users/kian/code/nhe-enga/pydicate
OK: import tupi -> /Users/kian/code/nhe-enga/tupi/tupi/__init__.py
OK: from pydicate.lang.tupilang import *
OK: from pydicate.lang.tupilang.pos import *


## 2. Build or load corpus data

The notebook uses the existing tokenizer scripts as data producers, not as a black-box final tokenizer.

Expected artifacts:

- `tokenizer/output/corpus.jsonl`
- `tokenizer/output/canonical_io.jsonl`
- `tokenizer/output/annotated_tokens.json`
- `tokenizer/output/annotated_tags.json`
- `tokenizer/output/annotated_subtags.json`
- optional `tokenizer/output/annotated_token_variants.json`

If the first two are missing, this cell runs the existing scripts with the commands requested for this proof of concept.

In [3]:
OUT_DIR = ROOT / "tokenizer" / "output"
CORPUS_JSONL = OUT_DIR / "corpus.jsonl"
CANONICAL_IO = OUT_DIR / "canonical_io.jsonl"
TOKENS_JSON = OUT_DIR / "annotated_tokens.json"
TAGS_JSON = OUT_DIR / "annotated_tags.json"
SUBTAGS_JSON = OUT_DIR / "annotated_subtags.json"
TOKEN_PAIRS_JSON = OUT_DIR / "annotated_token_pairs.json"
VARIANTS_JSON = OUT_DIR / "annotated_token_variants.json"


def run_repo_command(args: list[str]) -> None:
    print("$", " ".join(args))
    result = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(args)}")


OUT_DIR.mkdir(parents=True, exist_ok=True)

if not CORPUS_JSONL.exists():
    run_repo_command([
        "python3",
        "tokenizer/build_corpus_json.py",
        "--out_jsonl",
        "tokenizer/output/corpus.jsonl",
        "--include-synthetic",
        "--label-from-annotated",
    ])
else:
    print(f"Found {CORPUS_JSONL.relative_to(ROOT)}")

required_rawgrammar_outputs = [CANONICAL_IO, TOKENS_JSON, TAGS_JSON, SUBTAGS_JSON]
if not all(path.exists() for path in required_rawgrammar_outputs):
    run_repo_command([
        "python3",
        "tokenizer/rawgrammarpair.py",
        "--in_json",
        "tokenizer/output/corpus.jsonl",
        "--out_dir",
        "tokenizer/output",
    ])
else:
    print("Found canonical IO and core registries")

for path in [CORPUS_JSONL, CANONICAL_IO, TOKENS_JSON, TAGS_JSON, SUBTAGS_JSON, TOKEN_PAIRS_JSON, VARIANTS_JSON]:
    print(f"{path.relative_to(ROOT)}: {'present' if path.exists() else 'missing'}")

Found tokenizer/output/corpus.jsonl
Found canonical IO and core registries
tokenizer/output/corpus.jsonl: present
tokenizer/output/canonical_io.jsonl: present
tokenizer/output/annotated_tokens.json: present
tokenizer/output/annotated_tags.json: present
tokenizer/output/annotated_subtags.json: present
tokenizer/output/annotated_token_pairs.json: present
tokenizer/output/annotated_token_variants.json: present


In [4]:
def load_jsonl(path: Path, limit: int | None = None) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def load_json(path: Path, default=None):
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_registry(path: Path, value_key: str):
    items = load_json(path, default=[])
    id_to_value = {item["id"]: item[value_key] for item in items if "id" in item and value_key in item}
    value_to_id = {value: key for key, value in id_to_value.items()}
    return items, id_to_value, value_to_id


corpus_rows = load_jsonl(CORPUS_JSONL)
canonical_rows = load_jsonl(CANONICAL_IO)
token_items, id_to_morpheme, morpheme_to_id = load_registry(TOKENS_JSON, "value")
tag_items, id_to_tag, tag_to_id = load_registry(TAGS_JSON, "tag")
subtag_items, id_to_subtag, subtag_to_id = load_registry(SUBTAGS_JSON, "subtag")
token_pairs = load_json(TOKEN_PAIRS_JSON, default=[])
token_variants = load_json(VARIANTS_JSON, default=[])

print(f"corpus rows: {len(corpus_rows)}")
print(f"canonical IO rows: {len(canonical_rows)}")
print(f"M registry entries: {len(id_to_morpheme)}")
print(f"T registry entries: {len(id_to_tag)}")
print(f"S registry entries: {len(id_to_subtag)}")
print(f"token pair rows: {len(token_pairs)}")
print(f"variant rows: {len(token_variants)}")

print("\ncorpus.jsonl examples")
for row in corpus_rows[:3]:
    pprint(row)

print("\ncanonical_io.jsonl examples")
for row in canonical_rows[:3]:
    pprint(row)

print("\nannotated_tokens.json examples")
pprint(token_items[:5])

print("\nannotated_tags.json examples")
pprint(tag_items[:5])

print("\nannotated_subtags.json examples")
pprint(subtag_items[:8])

if token_variants:
    print("\nannotated_token_variants.json examples")
    pprint(token_variants[:5])
else:
    print("\nNo annotated_token_variants.json present. The baseline will use direct M-token surfaces only.")

corpus rows: 100
canonical IO rows: 100
M registry entries: 218
T registry entries: 347
S registry entries: 117
token pair rows: 271
variant rows: 9195

corpus.jsonl examples
{'anotated': 'Santa '
             "Cruz[DEEPEST_NODE_7:DIRECT:OBJECT:PROPER_NOUN]r[DEEPEST_NODE_5:PLURIFORM_PREFIX:R]a'ang[DEEPEST_NODE_6:ROOT]ab[DEEPEST_NODE_5:FACILITY_SUFFIX]a[CONSONANT_ENDING:DEEPEST_NODE_5:NOUN:POSSESSOR:SUBSTANTIVE_SUFFIX] "
             'r[DEEPEST_NODE_4:PLURIFORM_PREFIX:R]esé[DEEPEST_NODE_4:POSTPOSITION] '
             'oré[1ppe:DEEPEST_NODE_3:OBJECT:PRONOUN]pysyrõ[DEEPEST_NODE_1:ROOT] '
             'îepé[2ps:DEEPEST_NODE_1:OBJECT_1P:PRONOUN:SUBJECT] '
             'Tupã[DEEPEST_NODE_9:PROPER_NOUN] '
             'oré[1ppe:DEEPEST_NODE_11:OBJECT:POSSESSIVE_PRONOUN:PRONOUN] '
             'îar[DEEPEST_NODE_10:ROOT:VOCATIVE] '
             "oré[1ppe:DEEPEST_NODE_15:OBJECT:PRONOUN]amotar[DEEPEST_NODE_14:ROOT]e'ym[DEEPEST_NODE_13:NEGATION_SUFFIX]bar[ABSOLUTE_AGENT_SUFFIX:DEEPEST_NODE_13]a[CO

## 3. A cleaner morphology-token representation

The existing canonical stream is intentionally stable and compact:

```text
M000001 T000080 S000090 S000002 ...
```

For model training and inspection that is too opaque. This notebook converts it into a factorized stream:

- `M000001` becomes `<M:000001>`
- a full grammar tag like `[SUBJECT:1ps]` becomes `<G:SUBJECT> <G:1ps>`
- generated structure/debug features such as `DEEPEST_NODE_*`, `ROOT`, and `DIRECT` are dropped by default

The `S` IDs are usually derived from the `T` tag, so the default conversion expands `T` tags and skips `S` IDs to avoid double-counting.

In [5]:
DROP_FEATURE_PREFIXES = {"DEEPEST_NODE"}
DROP_FEATURES = {"ROOT", "DIRECT"}
USE_EXPLICIT_S_IDS = False


def split_canonical_id(tok: str) -> tuple[str, str] | None:
    m = re.fullmatch(r"([MTS])(\d{6})", tok)
    if not m:
        return None
    return m.group(1), m.group(2)


def is_dropped_feature(feature: str) -> bool:
    if feature in DROP_FEATURES:
        return True
    return any(feature.startswith(prefix) for prefix in DROP_FEATURE_PREFIXES)


def tag_to_grammar_features(tag: str) -> list[str]:
    inner = tag.strip()
    if inner.startswith("[") and inner.endswith("]"):
        inner = inner[1:-1]
    parts = [part for part in inner.split(":") if part]
    return [part for part in parts if not is_dropped_feature(part)]


def canonical_ids_to_factorized_tokens(
    canonical_output: str,
    id_to_morpheme: dict[str, str],
    id_to_tag: dict[str, str],
    id_to_subtag: dict[str, str],
) -> list[str]:
    out = []
    for tok in canonical_output.split():
        parsed = split_canonical_id(tok)
        if not parsed:
            out.append(tok)
            continue
        kind, digits = parsed
        if kind == "M":
            out.append(f"<M:{digits}>")
        elif kind == "T":
            tag = id_to_tag.get(tok)
            if tag is None:
                out.append(f"<T_UNKNOWN:{digits}>")
                continue
            out.extend(f"<G:{feature}>" for feature in tag_to_grammar_features(tag))
        elif kind == "S" and USE_EXPLICIT_S_IDS:
            subtag = id_to_subtag.get(tok)
            if subtag and not is_dropped_feature(subtag):
                out.append(f"<G:{subtag}>")
    return out


def morpheme_surface_from_factor(tok: str) -> str:
    if tok.startswith("<M:") and tok.endswith(">"):
        mid = "M" + tok[3:-1]
        return id_to_morpheme.get(mid, mid)
    if tok.startswith("<RAW:") and tok.endswith(">"):
        return tok[5:-1]
    return tok


def inspect_factorized(tokens: list[str]) -> list[dict]:
    groups = []
    current = None

    def flush():
        nonlocal current
        if current is not None:
            groups.append(current)
            current = None

    for tok in tokens:
        if tok.startswith("<M:") and tok.endswith(">"):
            flush()
            mid = "M" + tok[3:-1]
            current = {"token": tok, "surface": id_to_morpheme.get(mid, mid), "grammar": []}
        elif tok.startswith("<RAW:"):
            flush()
            groups.append({"token": tok, "surface": morpheme_surface_from_factor(tok), "grammar": ["RAW"]})
        elif tok.startswith("<G:") and tok.endswith(">"):
            feature = tok[3:-1]
            if current is None:
                current = {"token": None, "surface": None, "grammar": []}
            current["grammar"].append(feature)
        else:
            flush()
            groups.append({"token": tok, "surface": tok, "grammar": []})
    flush()
    return groups


def detokenize_factorized(tokens: list[str]) -> str:
    # Spacing is not fully reversible from the factorized stream; this is an inspection surface.
    return " ".join(group["surface"] for group in inspect_factorized(tokens) if group.get("surface"))


def compact_tokens(tokens: list[str], max_tokens: int = 80) -> str:
    if len(tokens) <= max_tokens:
        return " ".join(tokens)
    head = " ".join(tokens[:max_tokens])
    return f"{head} ... (+{len(tokens) - max_tokens} tokens)"


def pretty_compare(input_text: str, predicted: list[str], gold: list[str] | None = None, max_tokens: int = 80) -> None:
    print("INPUT:", input_text)
    if gold is not None:
        print("GOLD: ", compact_tokens(gold, max_tokens=max_tokens))
    print("PRED: ", compact_tokens(predicted, max_tokens=max_tokens))
    print("INSPECT:")
    for group in inspect_factorized(predicted)[:30]:
        print("  ", group)
    if len(inspect_factorized(predicted)) > 30:
        print("  ...")


factorized_examples = []
for row in canonical_rows:
    factorized_examples.append({
        "input": row["input"],
        "target": canonical_ids_to_factorized_tokens(row["output"], id_to_morpheme, id_to_tag, id_to_subtag),
        "canonical": row["output"],
    })

for ex in factorized_examples[:3]:
    print("\n---")
    pretty_compare(ex["input"], ex["target"], max_tokens=100)


---
INPUT: Santa Cruzra'angaba resé orépysyrõ îepé Tupã oré îar oréamotare'ymbara suí
PRED:  <M:000001> <M:000002> <M:000003> <G:PLURIFORM_PREFIX> <G:R> <M:000004> <M:000005> <G:FACILITY_SUFFIX> <M:000006> <G:CONSONANT_ENDING> <G:NOUN> <G:POSSESSOR> <G:SUBSTANTIVE_SUFFIX> <M:000003> <G:PLURIFORM_PREFIX> <G:R> <M:000007> <G:POSTPOSITION> <M:000008> <G:1ppe> <G:OBJECT> <G:PRONOUN> <M:000009> <M:000010> <G:2ps> <G:OBJECT_1P> <G:PRONOUN> <G:SUBJECT> <M:000011> <G:PROPER_NOUN> <M:000008> <G:1ppe> <G:OBJECT> <G:POSSESSIVE_PRONOUN> <G:PRONOUN> <M:000012> <M:000008> <G:1ppe> <G:OBJECT> <G:PRONOUN> <M:000013> <M:000014> <G:NEGATION_SUFFIX> <M:000015> <G:ABSOLUTE_AGENT_SUFFIX> <M:000006> <G:CONSONANT_ENDING> <G:SUBSTANTIVE_SUFFIX> <M:000016> <G:POSTPOSITION>
INSPECT:
   {'token': '<M:000001>', 'surface': 'Santa', 'grammar': []}
   {'token': '<M:000002>', 'surface': 'Cruz', 'grammar': []}
   {'token': '<M:000003>', 'surface': 'r', 'grammar': ['PLURIFORM_PREFIX', 'R']}
   {'token': '<M:000004>', 

## 4. Non-neural morphology baseline

Before training a model, this cell implements a transparent registry baseline:

- normalize text with Unicode NFC and whitespace cleanup
- use `annotated_token_variants.json` when present to map orthographic/allomorphic variants to canonical M IDs
- greedily segment each whitespace token by longest known morpheme surface
- emit `<RAW:...>` for unknown chunks
- attach the most frequent grammar feature sequence seen for each M ID in `canonical_io.jsonl`

This is deliberately simple. It gives us a registry/Viterbi-style comparison point before introducing a neural model.

In [6]:
def normalize_surface(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


surface_to_mids = defaultdict(list)
for mid, surface in id_to_morpheme.items():
    surface_to_mids[surface].append(mid)

variant_surface_to_canonical_ids = defaultdict(list)
variant_id_to_canonical_id = {}
stale_variant_rows = 0
for item in token_variants:
    variant = item.get("variant")
    canonical_id = item.get("canonical_id")
    variant_id = item.get("variant_id")

    # Variant files can outlive a previous, larger M registry. Keep only IDs that the
    # current annotated_tokens.json can actually inspect/detokenize.
    target_id = None
    if canonical_id in id_to_morpheme:
        target_id = canonical_id
    elif variant_id in id_to_morpheme:
        target_id = variant_id
    else:
        stale_variant_rows += 1

    if variant and target_id:
        variant_surface_to_canonical_ids[variant].append(target_id)
    if variant_id in id_to_morpheme and target_id:
        variant_id_to_canonical_id[variant_id] = target_id

if token_variants:
    kept = sum(len(v) for v in variant_surface_to_canonical_ids.values())
    print(f"variant candidates kept: {kept}; stale rows ignored: {stale_variant_rows}")

# Candidate surfaces include direct morpheme surfaces and optional variant surfaces.
surface_candidates = defaultdict(list)
for surface, mids in surface_to_mids.items():
    surface_candidates[surface].extend(mids)
for surface, mids in variant_surface_to_canonical_ids.items():
    surface_candidates[surface].extend(mids)

# Tie-break repeated candidates by M frequency in canonical_io.
m_frequency = Counter()
m_to_tagseq_counts = defaultdict(Counter)

for row in canonical_rows:
    current_m = None
    current_tags = []

    def flush_current():
        if current_m is None:
            return
        canonical_m = variant_id_to_canonical_id.get(current_m, current_m)
        m_frequency[canonical_m] += 1
        m_to_tagseq_counts[canonical_m][tuple(current_tags)] += 1

    for tok in row["output"].split():
        if tok.startswith("M"):
            flush_current()
            current_m = tok
            current_tags = []
        elif tok.startswith("T") and current_m is not None:
            current_tags.append(tok)
    flush_current()

for mid in id_to_morpheme:
    m_frequency.setdefault(mid, 0)


def best_mid_for_surface(surface: str) -> str | None:
    mids = surface_candidates.get(surface)
    if not mids:
        return None
    return max(set(mids), key=lambda mid: (m_frequency[mid], -len(id_to_morpheme.get(mid, "")), mid))


known_surfaces_by_first = defaultdict(list)
for surface in surface_candidates:
    if surface:
        known_surfaces_by_first[surface[0]].append(surface)
for first in known_surfaces_by_first:
    known_surfaces_by_first[first].sort(key=lambda s: (len(s), s), reverse=True)


def longest_match_at(word: str, i: int) -> tuple[str, str] | None:
    for surface in known_surfaces_by_first.get(word[i], []):
        if word.startswith(surface, i):
            mid = best_mid_for_surface(surface)
            if mid:
                return surface, mid
    return None


def segment_word_longest(word: str) -> list[str]:
    pieces = []
    i = 0
    raw_buffer = []

    def flush_raw():
        nonlocal raw_buffer
        if raw_buffer:
            pieces.append("<RAW:" + "".join(raw_buffer) + ">")
            raw_buffer = []

    while i < len(word):
        match = longest_match_at(word, i)
        if match is None:
            raw_buffer.append(word[i])
            i += 1
            continue
        flush_raw()
        surface, mid = match
        pieces.append(variant_id_to_canonical_id.get(mid, mid))
        i += len(surface)
    flush_raw()
    return pieces


def best_tag_sequence_for_mid(mid: str) -> tuple[str, ...]:
    counts = m_to_tagseq_counts.get(mid)
    if not counts:
        return ()
    return max(counts.items(), key=lambda item: (item[1], len(item[0]), item[0]))[0]


def grammar_tokens_for_tag_id(tid: str) -> list[str]:
    tag = id_to_tag.get(tid)
    if not tag:
        return []
    return [f"<G:{feature}>" for feature in tag_to_grammar_features(tag)]


def baseline_tokenize(text: str, include_grammar: bool = True) -> list[str]:
    text = normalize_surface(text)
    out = []
    for word in text.split(" ") if text else []:
        for piece in segment_word_longest(word):
            if piece.startswith("<RAW:"):
                out.append(piece)
                continue
            out.append("<M:" + piece[1:] + ">")
            if include_grammar:
                for tid in best_tag_sequence_for_mid(piece):
                    out.extend(grammar_tokens_for_tag_id(tid))
    return out


def raw_token_rate(tokens: list[str]) -> float:
    if not tokens:
        return 0.0
    raw = sum(1 for tok in tokens if tok.startswith("<RAW:"))
    m_or_raw = sum(1 for tok in tokens if tok.startswith("<M:") or tok.startswith("<RAW:"))
    return raw / max(1, m_or_raw)


for text in [
    "amém",
    "tuba ta'yra Espírito Santo rera pupé",
    "orépysyrõte îepé mba'eaíba suí",
    "xe rera",
    "xerera",
]:
    print("\n---")
    pred = baseline_tokenize(text)
    pretty_compare(text, pred, max_tokens=100)
    print(f"RAW token rate: {raw_token_rate(pred):.2%}")

variant candidates kept: 1201; stale rows ignored: 7994

---
INPUT: amém
PRED:  <M:000024> <G:AMEN> <G:INTERJECTION>
INSPECT:
   {'token': '<M:000024>', 'surface': 'amém', 'grammar': ['AMEN', 'INTERJECTION']}
RAW token rate: 0.00%

---
INPUT: tuba ta'yra Espírito Santo rera pupé
PRED:  <M:000017> <G:PERMISSIVE_PREFIX> <G:VOWEL> <M:000018> <M:000006> <M:000017> <G:PERMISSIVE_PREFIX> <G:VOWEL> <M:000019> <M:000006> <M:000020> <M:000021> <M:000003> <G:PLURIFORM_PREFIX> <G:R> <M:000022> <M:000006> <M:000023> <G:POSTPOSITION>
INSPECT:
   {'token': '<M:000017>', 'surface': 't', 'grammar': ['PERMISSIVE_PREFIX', 'VOWEL']}
   {'token': '<M:000018>', 'surface': 'ub', 'grammar': []}
   {'token': '<M:000006>', 'surface': 'a', 'grammar': []}
   {'token': '<M:000017>', 'surface': 't', 'grammar': ['PERMISSIVE_PREFIX', 'VOWEL']}
   {'token': '<M:000019>', 'surface': "a'yr", 'grammar': []}
   {'token': '<M:000006>', 'surface': 'a', 'grammar': []}
   {'token': '<M:000020>', 'surface': 'Espírito', 'gramm

## 5. Training data for a small seq2seq model

The neural model maps:

```text
surface input -> factorized morphology stream
```

By default it trains on at most `MAX_EXAMPLES = 3000`, but this checkout may have fewer current rows unless you regenerate with more sources or orthographic expansion. Adjust these values in the cell below before training.

In [7]:
MAX_EXAMPLES = 3000
EPOCHS = 3
BATCH_SIZE = 16
EMBED_DIM = 64
HIDDEN_DIM = 128
MAX_SRC_LEN = 220
MAX_TGT_LEN = 320
MAX_DECODE_LEN = 160
LEARNING_RATE = 3e-3
TEACHER_FORCING = 0.85
RANDOM_SEED = 7

PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
UNK = "<UNK>"

random.seed(RANDOM_SEED)

all_examples = [
    ex for ex in factorized_examples
    if len(ex["input"]) <= MAX_SRC_LEN and len(ex["target"]) + 2 <= MAX_TGT_LEN
]
all_examples = all_examples[:MAX_EXAMPLES]
random.shuffle(all_examples)

split_at = max(1, int(len(all_examples) * 0.8)) if len(all_examples) > 1 else len(all_examples)
train_examples = all_examples[:split_at]
dev_examples = all_examples[split_at:] or all_examples[:min(5, len(all_examples))]

src_chars = sorted({ch for ex in train_examples for ch in ex["input"]})
tgt_tokens = sorted({tok for ex in train_examples for tok in ex["target"]})

src_itos = [PAD, SOS, EOS, UNK] + src_chars
src_stoi = {tok: i for i, tok in enumerate(src_itos)}
tgt_itos = [PAD, SOS, EOS, UNK] + tgt_tokens
tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}

SRC_PAD_IDX = src_stoi[PAD]
TGT_PAD_IDX = tgt_stoi[PAD]


def encode_source(text: str) -> list[int]:
    text = normalize_surface(text)[:MAX_SRC_LEN]
    ids = [src_stoi[SOS]]
    ids.extend(src_stoi.get(ch, src_stoi[UNK]) for ch in text)
    ids.append(src_stoi[EOS])
    return ids[:MAX_SRC_LEN + 2]


def encode_target(tokens: list[str]) -> list[int]:
    clipped = tokens[: MAX_TGT_LEN - 2]
    ids = [tgt_stoi[SOS]]
    ids.extend(tgt_stoi.get(tok, tgt_stoi[UNK]) for tok in clipped)
    ids.append(tgt_stoi[EOS])
    return ids


def decode_target(ids: list[int]) -> list[str]:
    out = []
    for idx in ids:
        tok = tgt_itos[int(idx)]
        if tok == EOS:
            break
        if tok not in {PAD, SOS}:
            out.append(tok)
    return out


print(f"usable examples: {len(all_examples)}")
print(f"train/dev: {len(train_examples)} / {len(dev_examples)}")
print(f"source char vocab: {len(src_itos)}")
print(f"target token vocab: {len(tgt_itos)}")
if all_examples:
    print("\nexample input:", all_examples[0]["input"])
    print("example target:", compact_tokens(all_examples[0]["target"], max_tokens=80))
else:
    print("No examples survived the length filters. Raise MAX_SRC_LEN/MAX_TGT_LEN or rebuild the corpus.")

usable examples: 100
train/dev: 80 / 20
source char vocab: 53
target token vocab: 293

example input: 'ara mosapyra pupé omanõba'epûera suí sekobeîebyri
example target: <M:000047> <M:000006> <G:CONSONANT_ENDING> <G:NOUN> <G:SUBSTANTIVE_SUFFIX> <M:000162> <G:CARDINAL> <G:NUMBER> <G:THREE> <M:000023> <G:POSTPOSITION> <M:000035> <G:3p> <G:SUBJECT_PREFIX> <M:000163> <M:000070> <G:RELATIVE_AGENT_SUFFIX> <M:000148> <G:PRETERITE_SUFFIX> <M:000006> <G:CONSONANT_ENDING> <G:SUBSTANTIVE_SUFFIX> <M:000016> <G:POSTPOSITION> <M:000097> <G:PLURIFORM_PREFIX> <G:S> <M:000164> <M:000030> <G:CIRCUMSTANTIAL_SUFFIX> <G:CONSONANT_ENDING>


## 6. Small PyTorch encoder-decoder with attention

This is intentionally small and local. It does not download a pretrained model.

If `torch` is not installed, this cell prints the install command and leaves the baseline usable:

```bash
python3 -m pip install torch numpy notebook ipykernel
```

In [8]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
    print(f"torch: {torch.__version__}")
except ModuleNotFoundError:
    TORCH_AVAILABLE = False
    print("Torch is not installed. Install it to run neural training:")
    print("  python3 -m pip install torch numpy notebook ipykernel")

NEURAL_READY = False
neural_model = None

if TORCH_AVAILABLE and train_examples:
    def pad_sequences(seqs: list[list[int]], pad_idx: int):
        max_len = max(len(seq) for seq in seqs)
        tensor = torch.full((len(seqs), max_len), pad_idx, dtype=torch.long)
        for i, seq in enumerate(seqs):
            tensor[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        return tensor

    def make_batches(examples: list[dict], batch_size: int, shuffle: bool = True):
        rows = examples[:]
        if shuffle:
            random.shuffle(rows)
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            src = pad_sequences([encode_source(ex["input"]) for ex in batch], SRC_PAD_IDX)
            tgt = pad_sequences([encode_target(ex["target"]) for ex in batch], TGT_PAD_IDX)
            yield src, tgt, batch

    class Encoder(nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pad_idx: int):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
            self.hidden_proj = nn.Linear(hidden_dim * 2, hidden_dim)

        def forward(self, src):
            emb = self.embedding(src)
            outputs, hidden = self.gru(emb)
            hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)
            hidden0 = torch.tanh(self.hidden_proj(hidden_cat)).unsqueeze(0)
            return outputs, hidden0

    class AttentionDecoder(nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, enc_dim: int, pad_idx: int):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.enc_proj = nn.Linear(enc_dim, hidden_dim)
            self.gru = nn.GRU(embed_dim + enc_dim, hidden_dim, batch_first=True)
            self.out = nn.Linear(hidden_dim + enc_dim + embed_dim, vocab_size)

        def step(self, prev_tok, hidden, enc_outputs, src_mask):
            emb = self.embedding(prev_tok).unsqueeze(1)
            projected = self.enc_proj(enc_outputs)
            scores = torch.bmm(projected, hidden[-1].unsqueeze(2)).squeeze(2)
            scores = scores.masked_fill(~src_mask, -1e9)
            weights = torch.softmax(scores, dim=1).unsqueeze(1)
            context = torch.bmm(weights, enc_outputs)
            gru_input = torch.cat([emb, context], dim=2)
            output, hidden = self.gru(gru_input, hidden)
            logits = self.out(torch.cat([output.squeeze(1), context.squeeze(1), emb.squeeze(1)], dim=1))
            return logits, hidden

    class Seq2Seq(nn.Module):
        def __init__(self, src_vocab: int, tgt_vocab: int, embed_dim: int, hidden_dim: int):
            super().__init__()
            self.encoder = Encoder(src_vocab, embed_dim, hidden_dim, SRC_PAD_IDX)
            self.decoder = AttentionDecoder(tgt_vocab, embed_dim, hidden_dim, hidden_dim * 2, TGT_PAD_IDX)

        def forward(self, src, tgt, teacher_forcing: float = 1.0):
            src_mask = src != SRC_PAD_IDX
            enc_outputs, hidden = self.encoder(src)
            prev_tok = tgt[:, 0]
            logits_by_t = []
            for t in range(1, tgt.size(1)):
                logits, hidden = self.decoder.step(prev_tok, hidden, enc_outputs, src_mask)
                logits_by_t.append(logits.unsqueeze(1))
                predicted = logits.argmax(dim=1)
                if random.random() < teacher_forcing:
                    prev_tok = tgt[:, t]
                else:
                    prev_tok = predicted
            return torch.cat(logits_by_t, dim=1)

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print("device:", device)

    neural_model = Seq2Seq(len(src_itos), len(tgt_itos), EMBED_DIM, HIDDEN_DIM).to(device)
    optimizer = torch.optim.Adam(neural_model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

    for epoch in range(1, EPOCHS + 1):
        neural_model.train()
        losses = []
        for src, tgt, _batch in make_batches(train_examples, BATCH_SIZE, shuffle=True):
            src = src.to(device)
            tgt = tgt.to(device)
            optimizer.zero_grad()
            logits = neural_model(src, tgt, teacher_forcing=TEACHER_FORCING)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(neural_model.parameters(), 1.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"epoch {epoch:02d} train_loss={sum(losses) / max(1, len(losses)):.4f}")

    @torch.no_grad()
    def neural_predict(text: str, max_len: int | None = None) -> list[str]:
        neural_model.eval()
        if max_len is None:
            max_len = min(MAX_TGT_LEN, MAX_DECODE_LEN)
        src_ids = encode_source(text)
        src = torch.tensor([src_ids], dtype=torch.long, device=device)
        src_mask = src != SRC_PAD_IDX
        enc_outputs, hidden = neural_model.encoder(src)
        prev_tok = torch.tensor([tgt_stoi[SOS]], dtype=torch.long, device=device)
        out_ids = []
        repeated_suffix_counts = Counter()
        for _ in range(max_len):
            logits, hidden = neural_model.decoder.step(prev_tok, hidden, enc_outputs, src_mask)
            next_id = int(logits.argmax(dim=1).item())
            if tgt_itos[next_id] == EOS:
                break
            out_ids.append(next_id)

            # Weak early models often fall into loops before they learn EOS. Stop the
            # demo decode when the same short suffix repeats, while keeping the raw
            # training objective unchanged.
            if len(out_ids) >= 18:
                suffix = tuple(out_ids[-6:])
                repeated_suffix_counts[suffix] += 1
                if repeated_suffix_counts[suffix] >= 3:
                    out_ids = out_ids[:-6]
                    break

            prev_tok = torch.tensor([next_id], dtype=torch.long, device=device)
        return decode_target(out_ids)

    NEURAL_READY = True
    print("Neural model trained. Showing held-out predictions:")
    for ex in dev_examples[:5]:
        print("\n---")
        pretty_compare(ex["input"], neural_predict(ex["input"]), gold=ex["target"], max_tokens=80)
elif TORCH_AVAILABLE:
    print("No train examples are available after filtering; adjust MAX_* values or rebuild data.")

torch: 2.11.0
device: mps
epoch 01 train_loss=5.1271
epoch 02 train_loss=4.2025
epoch 03 train_loss=3.5651
Neural model trained. Showing held-out predictions:

---
INPUT: amém Jesus
GOLD:  <M:000024> <G:AMEN> <G:INTERJECTION> <M:000082> <G:PROPER_NOUN>
PRED:  <M:000024> <G:AMEN> <G:INTERJECTION>
INSPECT:
   {'token': '<M:000024>', 'surface': 'amém', 'grammar': ['AMEN', 'INTERJECTION']}

---
INPUT: Poncio Pilato morubixabamo sekóreme serekomemûãmbyramo sekóû
GOLD:  <M:000150> <M:000151> <M:000152> <M:000033> <G:POSTPOSITION> <G:SIMULATIVE_SUFFIX> <G:TRANSLATIONAL> <M:000097> <G:PLURIFORM_PREFIX> <G:S> <M:000153> <M:000154> <G:CONJUNCTIVE_SUFFIX> <M:000097> <G:PLURIFORM_PREFIX> <G:S> <M:000057> <M:000155> <G:AGENTLESS_PATIENT_SUFFIX> <M:000033> <G:POSTPOSITION> <G:SIMULATIVE_SUFFIX> <G:TRANSLATIONAL> <M:000097> <G:PLURIFORM_PREFIX> <G:S> <M:000074> <M:000075> <G:CIRCUMSTANTIAL_SUFFIX> <G:VOWEL_ENDING>
PRED:  <M:000006> <G:CONSONANT_ENDING> <G:SUBSTANTIVE_SUFFIX> <G:SUBSTANTIVE_SUFFIX> <G

## 7. Evaluation

These metrics are intentionally simple and readable:

- exact sequence match
- multiset token precision / recall / F1
- position-wise token accuracy
- morpheme-token sequence exact match
- baseline raw-token rate

The dataset is small by default, so treat these as sanity checks rather than a final benchmark.

In [9]:
def sequence_exact(pred: list[str], gold: list[str]) -> float:
    return float(pred == gold)


def token_prf(pred: list[str], gold: list[str]) -> tuple[float, float, float]:
    pred_c = Counter(pred)
    gold_c = Counter(gold)
    overlap = sum((pred_c & gold_c).values())
    precision = overlap / max(1, sum(pred_c.values()))
    recall = overlap / max(1, sum(gold_c.values()))
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def token_accuracy(pred: list[str], gold: list[str]) -> float:
    denom = max(len(pred), len(gold), 1)
    same = sum(1 for a, b in zip(pred, gold) if a == b)
    return same / denom


def m_tokens(tokens: list[str]) -> list[str]:
    return [tok for tok in tokens if tok.startswith("<M:") or tok.startswith("<RAW:")]


def evaluate_prediction_fn(name: str, pred_fn, examples: list[dict], limit: int = 50) -> dict:
    rows = examples[:limit]
    exacts = []
    precisions = []
    recalls = []
    f1s = []
    accuracies = []
    m_exact = []
    raw_rates = []
    for ex in rows:
        pred = pred_fn(ex["input"])
        gold = ex["target"]
        exacts.append(sequence_exact(pred, gold))
        p, r, f1 = token_prf(pred, gold)
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)
        accuracies.append(token_accuracy(pred, gold))
        m_exact.append(sequence_exact(m_tokens(pred), m_tokens(gold)))
        raw_rates.append(raw_token_rate(pred))
    result = {
        "name": name,
        "n": len(rows),
        "exact_sequence": sum(exacts) / max(1, len(exacts)),
        "token_precision": sum(precisions) / max(1, len(precisions)),
        "token_recall": sum(recalls) / max(1, len(recalls)),
        "token_f1": sum(f1s) / max(1, len(f1s)),
        "position_token_accuracy": sum(accuracies) / max(1, len(accuracies)),
        "morpheme_sequence_exact": sum(m_exact) / max(1, len(m_exact)),
        "raw_token_rate": sum(raw_rates) / max(1, len(raw_rates)),
    }
    return result


eval_examples = dev_examples or all_examples

baseline_eval = evaluate_prediction_fn("baseline", baseline_tokenize, eval_examples, limit=50)
pprint(baseline_eval)

if NEURAL_READY:
    neural_eval = evaluate_prediction_fn("neural", neural_predict, eval_examples, limit=50)
    pprint(neural_eval)
else:
    print("Neural evaluation skipped because the model has not been trained in this kernel.")

{'exact_sequence': 0.2,
 'morpheme_sequence_exact': 0.6,
 'n': 20,
 'name': 'baseline',
 'position_token_accuracy': 0.33737026549056626,
 'raw_token_rate': 0.027786449661449664,
 'token_f1': 0.808472247773644,
 'token_precision': 0.8191266270119456,
 'token_recall': 0.8140292293095497}
{'exact_sequence': 0.0,
 'morpheme_sequence_exact': 0.0,
 'n': 20,
 'name': 'neural',
 'position_token_accuracy': 0.05350889377743597,
 'raw_token_rate': 0.0,
 'token_f1': 0.2000590631757319,
 'token_precision': 0.2896642883023957,
 'token_recall': 0.18279576216647156}


In [10]:
print("Qualitative comparison")
for ex in (dev_examples or all_examples)[:5]:
    print("\n==============================")
    print("INPUT:", ex["input"])
    print("GOLD:    ", compact_tokens(ex["target"], max_tokens=90))
    base = baseline_tokenize(ex["input"])
    print("BASELINE:", compact_tokens(base, max_tokens=90), f"raw_rate={raw_token_rate(base):.1%}")
    if NEURAL_READY:
        neural = neural_predict(ex["input"])
        print("NEURAL:  ", compact_tokens(neural, max_tokens=90))
    else:
        print("NEURAL:   skipped; train the neural cell after installing torch")

Qualitative comparison

INPUT: amém Jesus
GOLD:     <M:000024> <G:AMEN> <G:INTERJECTION> <M:000082> <G:PROPER_NOUN>
BASELINE: <M:000024> <G:AMEN> <G:INTERJECTION> <M:000082> <G:PROPER_NOUN> raw_rate=0.0%
NEURAL:   <M:000024> <G:AMEN> <G:INTERJECTION>

INPUT: Poncio Pilato morubixabamo sekóreme serekomemûãmbyramo sekóû
GOLD:     <M:000150> <M:000151> <M:000152> <M:000033> <G:POSTPOSITION> <G:SIMULATIVE_SUFFIX> <G:TRANSLATIONAL> <M:000097> <G:PLURIFORM_PREFIX> <G:S> <M:000153> <M:000154> <G:CONJUNCTIVE_SUFFIX> <M:000097> <G:PLURIFORM_PREFIX> <G:S> <M:000057> <M:000155> <G:AGENTLESS_PATIENT_SUFFIX> <M:000033> <G:POSTPOSITION> <G:SIMULATIVE_SUFFIX> <G:TRANSLATIONAL> <M:000097> <G:PLURIFORM_PREFIX> <G:S> <M:000074> <M:000075> <G:CIRCUMSTANTIAL_SUFFIX> <G:VOWEL_ENDING>
BASELINE: <M:000150> <M:000151> <M:000152> <M:000033> <G:POSTPOSITION> <G:SIMULATIVE_SUFFIX> <G:TRANSLATIONAL> <M:000200> <M:000003> <G:PLURIFORM_PREFIX> <G:R> <M:000154> <G:CONJUNCTIVE_SUFFIX> <M:000097> <G:PLURIFORM_PREFIX> 

## 8. Manual test strings

Edit `TEST_STRINGS` and rerun this cell. It shows the baseline, the neural model if trained, and the inspection view.

In [11]:
TEST_STRINGS = [
    "amém",
    "tuba ta'yra Espírito Santo rera pupé",
    "orépysyrõte îepé mba'eaíba suí",
    "aîpotar nde kûara",
    "xe rera",
    "xerera",
]

for text in TEST_STRINGS:
    print("\n" + "=" * 80)
    baseline = baseline_tokenize(text)
    print("BASELINE")
    pretty_compare(text, baseline, max_tokens=120)
    print(f"baseline raw token rate: {raw_token_rate(baseline):.2%}")

    if NEURAL_READY:
        neural = neural_predict(text)
        print("\nNEURAL")
        pretty_compare(text, neural, max_tokens=120)
    else:
        print("\nNEURAL skipped; run the training cell after installing torch.")


BASELINE
INPUT: amém
PRED:  <M:000024> <G:AMEN> <G:INTERJECTION>
INSPECT:
   {'token': '<M:000024>', 'surface': 'amém', 'grammar': ['AMEN', 'INTERJECTION']}
baseline raw token rate: 0.00%

NEURAL
INPUT: amém
PRED:  <M:000024> <G:AMEN> <G:INTERJECTION>
INSPECT:
   {'token': '<M:000024>', 'surface': 'amém', 'grammar': ['AMEN', 'INTERJECTION']}

BASELINE
INPUT: tuba ta'yra Espírito Santo rera pupé
PRED:  <M:000017> <G:PERMISSIVE_PREFIX> <G:VOWEL> <M:000018> <M:000006> <M:000017> <G:PERMISSIVE_PREFIX> <G:VOWEL> <M:000019> <M:000006> <M:000020> <M:000021> <M:000003> <G:PLURIFORM_PREFIX> <G:R> <M:000022> <M:000006> <M:000023> <G:POSTPOSITION>
INSPECT:
   {'token': '<M:000017>', 'surface': 't', 'grammar': ['PERMISSIVE_PREFIX', 'VOWEL']}
   {'token': '<M:000018>', 'surface': 'ub', 'grammar': []}
   {'token': '<M:000006>', 'surface': 'a', 'grammar': []}
   {'token': '<M:000017>', 'surface': 't', 'grammar': ['PERMISSIVE_PREFIX', 'VOWEL']}
   {'token': '<M:000019>', 'surface': "a'yr", 'grammar':

## Limitations and next steps

This proof of concept is intentionally rough:

- spacing is not fully reversible from the factorized stream
- the baseline is greedy longest-match, not a full weighted transducer
- grammar features are projected from current `T` tags and may need a better feature ontology
- the neural model is tiny and trained from scratch; it needs more generated examples and orthographic variants to generalize
- exact match is harsh because many surface strings have legitimate ambiguous analyses

Useful next experiments:

- rebuild the corpus with more synthetic coverage and orthographic expansion
- compare the greedy baseline with the existing Viterbi scorer
- add explicit boundary tokens if detokenization fidelity matters
- add a constrained decoder so the neural model cannot invent invalid `<M:...>` IDs or malformed `<G:...>` tags